In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error
train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv")


/tmp/ipykernel_16/2239211371.py:7: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


In [2]:
train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv")


/tmp/ipykernel_16/1991816170.py:1: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


In [3]:
TARGET = "TargetValue"
ID_COLUMN = "TransactionID"

test_ids = test[ID_COLUMN].copy()

all_features = pd.concat(
    [train.drop(columns=[TARGET]), test],
    axis=0,
    ignore_index=True
)

date = pd.to_datetime(all_features["TransactionDate"], errors="coerce")

all_features["TransactionYear"] = date.dt.year
all_features["TransactionMonth"] = date.dt.month
all_features["TransactionQuarter"] = date.dt.quarter
all_features["TransactionDayOfYear"] = date.dt.dayofyear
all_features["TransactionDays"] = (date - pd.Timestamp("1990-01-01")).dt.days

all_features["ManufactureYear"] = pd.to_numeric(
    all_features["ManufactureYear"],
    errors="coerce"
)

all_features["ManufactureYear"] = all_features["ManufactureYear"].where(
    all_features["ManufactureYear"].between(1900, 2020)
)

all_features["AssetAge"] = all_features["TransactionYear"] - all_features["ManufactureYear"]
all_features["AssetAge"] = all_features["AssetAge"].where(
    all_features["AssetAge"].between(0, 100)
)

all_features["LogOperationalHours"] = np.log1p(
    pd.to_numeric(all_features["OperationalHoursMeter"], errors="coerce").clip(lower=0)
)

for col in [
    "ProductConfigID",
    "Spec_FullDescriptor",
    "Spec_BaseClass",
    "Spec_SubClass",
    "VendorPartnerID",
    "RegionCode"
]:
    all_features[f"{col}_Frequency"] = all_features[col].map(
        all_features[col].value_counts(dropna=False)
    ).astype("float32")

all_features = all_features.drop(
    columns=["AssetID", ID_COLUMN, "TransactionDate"],
    errors="ignore"
)

X = all_features.iloc[:len(train)].copy()
X_test = all_features.iloc[len(train):].copy()
y = np.log1p(train[TARGET])

target_encode_cols = [
    "ProductConfigID",
    "Spec_FullDescriptor",
    "Spec_BaseClass",
    "Spec_SubClass",
    "VendorPartnerID",
    "RegionCode",
    "InventoryGroupCategory",
    "InventoryGroupDescription"
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in target_encode_cols:
    if col not in X.columns:
        continue

    X[f"{col}_TE"] = np.nan
    X_test[f"{col}_TE"] = 0.0

    for train_idx, valid_idx in kf.split(X):
        stats = y.iloc[train_idx].groupby(X.iloc[train_idx][col]).mean()
        X.loc[X.index[valid_idx], f"{col}_TE"] = X.iloc[valid_idx][col].map(stats)

    full_stats = y.groupby(X[col]).mean()
    X_test[f"{col}_TE"] = X_test[col].map(full_stats)

    global_mean = y.mean()
    X[f"{col}_TE"] = X[f"{col}_TE"].fillna(global_mean).astype("float32")
    X_test[f"{col}_TE"] = X_test[f"{col}_TE"].fillna(global_mean).astype("float32")

categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

if "ProductConfigID" in X.columns:
    categorical_columns.append("ProductConfigID")

categorical_columns = list(dict.fromkeys(categorical_columns))

for col in categorical_columns:
    X[col] = X[col].astype(str).fillna("__MISSING__").astype("category")
    X_test[col] = X_test[col].astype(str).fillna("__MISSING__").astype("category")

params = {
    "objective": "regression",
    "metric": "rmse",
    "n_estimators": 6000,
    "learning_rate": 0.025,
    "num_leaves": 95,
    "min_child_samples": 30,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.05,
    "reg_lambda": 1.5,
    "cat_smooth": 30,
    "cat_l2": 15,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True
}

valid_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_columns,
        eval_set=[(X_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(200)
        ]
    )

    valid_predictions[valid_idx] = np.expm1(
        model.predict(X_valid, num_iteration=model.best_iteration_)
    )

    test_predictions += np.expm1(
        model.predict(X_test, num_iteration=model.best_iteration_)
    ) / kf.n_splits

    best_iterations.append(model.best_iteration_)

    fold_rmsle = np.sqrt(
        mean_squared_log_error(
            np.expm1(y_valid),
            np.clip(valid_predictions[valid_idx], 0, None)
        )
    )

    print(f"Fold {fold} RMSLE: {fold_rmsle:.6f}")

valid_predictions = np.clip(valid_predictions, 0, None)
test_predictions = np.clip(test_predictions, 0, None)

cv_rmsle = np.sqrt(
    mean_squared_log_error(
        train[TARGET],
        valid_predictions
    )
)

print("CV RMSLE:", cv_rmsle)
print("Average best iteration:", int(np.mean(best_iterations)))

submission[ID_COLUMN] = test_ids
submission[TARGET] = test_predictions

submission.to_csv("submission.csv", index=False)

print("Created submission.csv")
print(submission.head())

[200]	valid_0's rmse: 0.220203
[400]	valid_0's rmse: 0.210478
[600]	valid_0's rmse: 0.20715
[800]	valid_0's rmse: 0.20542
[1000]	valid_0's rmse: 0.204379
[1200]	valid_0's rmse: 0.203697
[1400]	valid_0's rmse: 0.203102
[1600]	valid_0's rmse: 0.202619
[1800]	valid_0's rmse: 0.202328
[2000]	valid_0's rmse: 0.202037
[2200]	valid_0's rmse: 0.201782
[2400]	valid_0's rmse: 0.201599
[2600]	valid_0's rmse: 0.201403
[2800]	valid_0's rmse: 0.201204
[3000]	valid_0's rmse: 0.201048
[3200]	valid_0's rmse: 0.200937
[3400]	valid_0's rmse: 0.200816
[3600]	valid_0's rmse: 0.200728
[3800]	valid_0's rmse: 0.200649
[4000]	valid_0's rmse: 0.200588
[4200]	valid_0's rmse: 0.200517
[4400]	valid_0's rmse: 0.200432
[4600]	valid_0's rmse: 0.200351
[4800]	valid_0's rmse: 0.200323
Fold 1 RMSLE: 0.200312
[200]	valid_0's rmse: 0.22216
[400]	valid_0's rmse: 0.212311
[600]	valid_0's rmse: 0.208641
[800]	valid_0's rmse: 0.206845
[1000]	valid_0's rmse: 0.205806
[1200]	valid_0's rmse: 0.204987
[1400]	valid_0's rmse: 0.204

# MILESTONE-1

In [4]:
print(train.shape,test.shape)

(138701, 50) (15000, 49)


In [5]:
set(train.columns)-set(test.columns)

{'TargetValue'}

In [6]:
cols=train.select_dtypes(include='number').columns
len(cols)


6

In [7]:
missing=train.isna().sum()
missing[missing > 124800]

col4     128320
col5     128320
col18    138635
col19    138635
dtype: int64

In [8]:
train['OperationalHoursMeter'].isna().mean()*100

np.float64(40.838205924975306)

In [9]:
train['TargetValue'].median()

35000.0

In [10]:
train['ManufactureYear'].mode()[0]

np.int64(1001)

In [11]:
train['TransactionDate']=pd.to_datetime(train['TransactionDate'])
train['TransactionDate'].min()

Timestamp('1990-01-31 00:00:00')

In [12]:
train['Spec_BaseClass'].nunique()

1249

In [13]:
train['RegionCode'].value_counts().head()

RegionCode
Florida       23460
Texas         18284
California     9525
Washington     6063
Georgia        5240
Name: count, dtype: int64

In [14]:
train[train['RegionCode']=='Florida']['TargetValue'].mean()

np.float64(45007.54262574595)

In [15]:
train['UtilizationTier'].value_counts(dropna=False)

UtilizationTier
NaN       88226
Medium    24588
Low       17005
High       8882
Name: count, dtype: int64

In [16]:
train[['TargetValue','OperationalHoursMeter']].corr()

,TargetValue,OperationalHoursMeter
TargetValue,1.000000,0.001994
OperationalHoursMeter,0.001994,1.000000


In [17]:
train['FunctionalClassification'].value_counts().head()

FunctionalClassification
Hydraulic Excavator, Track - 21.0 to 24.0 Metric Tons      11158
Motorgrader - 145.0 to 170.0 Horsepower                     8107
Hydraulic Excavator, Track - 12.0 to 14.0 Metric Tons       7625
Backhoe Loader - 14.0 to 15.0 Ft Standard Digging Depth     7603
Hydraulic Excavator, Track - 33.0 to 40.0 Metric Tons       7440
Name: count, dtype: int64